In [0]:
import dlt
import pyspark.sql.functions as F

Extract from Workspace and load it in Bronze Address as Streaming data

In [0]:
@dlt.table(
    name = 'bronze_addresses',
    table_properties = {'quality': 'bronze'},
    comment = 'Raw address data ingested from the source system'
           )

def create_bronze_addresses():
  return (spark.readStream.format('cloudFiles') 
                         .option('cloudFiles.format', 'csv') 
                         .option('cloudFiles.inferColumnTypes', True) 
                         .load('/Volumes/circuitbox/landing/operational_data/addresses')
                         .select("*",
                                 F.col("_metadata.file_path").alias("input_file_path"),
                                 F.col("current_timestamp").alias("ingested_timestamp")
                                 )
  )

In [0]:
@dlt.table(
    name = 'silver_addresses',
    table_properties = {'quality' : 'silver'},
    comment ='Silver Address table from Brozne table'
)
@dlt.expect_or_fail("valid_customer_id","customer_id is not null")
@dlt.expect_or_drop("valid_address","address_line_1 is not null")
@dlt.expect("valid_postcode","length(postcode) = 5")
def create_silver_addresses_clean ():
    return (
        spark.readStream.table('bronze_addresses')
        .select("customer_id","address_line_1","city","state","postcode",
                F.col("created_date").cast("date").alias("created_date")
                )
    )

In [0]:
dlt.create_streaming_live_table(
    name ="silver_addresse_final",
    comment = "SCD Type2 table",
    table_properties = {'quality' : 'silver'}
    )

In [0]:
apply_changes = dlt.apply_changes(
    target = "silver_addresse_final",
    source = "silver_addresses",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
    )